In [2]:
import pandas as pd
import sqlite3

In [ ]:
df = pd.read_csv('/home/aniketnerali16/new/WA_Fn-UseC_-Telco-Customer-Churn-Cleaned.csv')

conn = sqlite3.connect('/home/aniketnerali16/new/Churn.db')

df.to_sql('customers', conn, if_exists='replace', index = False )

print("Database Created")
print(f"Row Loaded: {len(df)}")



Database Created
Row Loaded: 7043


In [4]:
def run_query(query):
    return pd.read_sql_query(query,conn)

In [5]:
run_query("SELECT * FROM customers LIMIT 5")

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_num
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [6]:
run_query("""
    SELECT 
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(
            SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
        ) AS churn_rate_pct
    FROM customers
""")

,total_customers,churned,churn_rate_pct
0,7043,1869,26.5


In [7]:
run_query("""
    SELECT 
        Contract,
        COUNT(*) AS total,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned,
        ROUND(
            SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
        ) AS churn_rate_pct
    FROM customers
    GROUP BY Contract
    ORDER BY churn_rate_pct DESC
""")

,Contract,total,churned,churn_rate_pct
0,Month-to-month,3875,1655,42.7
1,One year,1473,166,11.3
2,Two year,1695,48,2.8


In [8]:
run_query("""
    SELECT 
        InternetService,
        COUNT(*) AS total,
        ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges,
        ROUND(
            SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
        ) AS churn_rate_pct
    FROM customers
    GROUP BY InternetService
    ORDER BY churn_rate_pct DESC
""")

,InternetService,total,avg_monthly_charges,churn_rate_pct
0,Fiber optic,3096,91.50,41.9
1,DSL,2421,58.10,19.0
2,No,1526,21.08,7.4


In [9]:
run_query("SELECT * FROM customers LIMIT 5")


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_num
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [13]:
run_query("""
    SELECT 
        CASE 
            WHEN tenure <= 12  THEN '0-12 months'
            WHEN tenure <= 24  THEN '13-24 months'
            WHEN tenure <= 48  THEN '25-48 months'
            ELSE '49+ months'
        END AS tenure_group,
        COUNT(*) AS total,
        ROUND(
            SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
        ) AS churn_rate_pct
    FROM customers
    GROUP BY tenure_group
    ORDER BY churn_rate_pct DESC
""")

,tenure_group,total,churn_rate_pct
0,0-12 months,2186,47.4
1,13-24 months,1024,28.7
2,25-48 months,1594,20.4
3,49+ months,2239,9.5


In [14]:
run_query("""
    SELECT 
        customerID,
        Contract,
        MonthlyCharges,
        Churn,
        RANK() OVER (
            PARTITION BY Contract 
            ORDER BY MonthlyCharges DESC
        ) AS charge_rank
    FROM customers
    LIMIT 20
""")

,customerID,Contract,MonthlyCharges,Churn,charge_rank
0,2302-ANTDP,Month-to-month,117.45,Yes,1
1,8016-NCFVO,Month-to-month,116.50,No,2
2,9659-QEQSY,Month-to-month,115.65,No,3
3,4361-BKAXE,Month-to-month,114.50,Yes,4
4,6710-HSJRD,Month-to-month,114.10,No,5
5,9158-VCTQB,Month-to-month,113.60,Yes,6
6,7279-BUYWN,Month-to-month,113.20,Yes,7
7,0771-WLCLA,Month-to-month,112.95,No,8
8,1583-IHQZE,Month-to-month,112.95,Yes,8
9,9481-IEBZY,Month-to-month,112.90,No,10


In [20]:
run_query("""
    SELECT 
        customerID,
        Contract,
        tenure,
        MonthlyCharges,
        InternetService,
        TechSupport,
        Churn
    FROM customers
    WHERE 
        Contract = 'Month-to-month'
        AND tenure <= 12
        AND MonthlyCharges > 65
        AND TechSupport = 'No'
    ORDER BY MonthlyCharges DESC
    LIMIT 20
""")


,customerID,Contract,tenure,MonthlyCharges,InternetService,TechSupport,Churn
0,5760-IFJOZ,Month-to-month,3,107.95,Fiber optic,No,No
1,2027-FECZV,Month-to-month,12,106.70,Fiber optic,No,Yes
2,1400-MMYXY,Month-to-month,3,105.90,Fiber optic,No,Yes
3,3389-YGYAI,Month-to-month,8,105.50,Fiber optic,No,Yes
4,5052-PNLOS,Month-to-month,3,105.35,Fiber optic,No,Yes
5,6734-GMPVK,Month-to-month,5,105.30,Fiber optic,No,No
6,7145-FEJWU,Month-to-month,12,105.30,Fiber optic,No,No
7,4587-VVTOX,Month-to-month,6,105.30,Fiber optic,No,Yes
8,6496-SLWHQ,Month-to-month,3,105.00,Fiber optic,No,Yes
9,5923-GXUOC,Month-to-month,10,104.40,Fiber optic,No,Yes


In [23]:
conn.execute("""
    CREATE VIEW IF NOT EXISTS churn_summary AS
    SELECT 
        Contract,
        InternetService,
        TechSupport,
        CASE 
            WHEN tenure <= 12 THEN '0-12 months'
            WHEN tenure <= 24 THEN '13-24 months'
            WHEN tenure <= 48 THEN '25-48 months'
            ELSE '49+ months'
        END AS tenure_group,
        COUNT(*) AS total_customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned_customers,
        ROUND(AVG(MonthlyCharges), 2) AS avg_monthly_charges,
        ROUND(
            SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
        ) AS churn_rate_pct
    FROM customers
    GROUP BY Contract, InternetService, TechSupport, tenure_group
""")

print("View created! This is what Power BI will connect to.")

View created! This is what Power BI will connect to.


In [24]:
summary_df = run_query("SELECT * FROM churn_summary")
summary_df.to_csv("/home/aniketnerali16/new/WA_Fn-UseC_-Telco-Customer-Churn-Cleaned-Summary.csv") 
print(f"Exported {len(summary_df)} rows to churn_summary.csv")

Exported 57 rows to churn_summary.csv
